In [ ]:
import torch
import torch.nn as nn
import timm
import torchvision
import torchvision.transforms as transforms
from huggingface_hub import hf_hub_download

# 1. SCARICA IL MODELLO ESATTO IN LOCALE
repo_id = "edadaltocg/resnet50_cifar10"
filename = "pytorch_model.bin"
local_dir = "./modello_edadaltocg" # Scegli la cartella che preferisci

print("1. Download/Verifica del file locale...")
percorso_locale = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=local_dir)
print(f"File pronto in: {percorso_locale}")

# 2. CREA L'ARCHITETTURA TIMM CORRETTA
print("\n2. Costruzione della ResNet-50 per CIFAR-10...")
model = timm.create_model('resnet50', pretrained=False, num_classes=10)

# Questa è la modifica vitale per le immagini 32x32 (quella che spaccava tutto prima)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()

# 3. CARICA I PESI DAL DISCO
print("\n3. Iniezione dei pesi...")
state_dict = torch.load(percorso_locale, map_location='cpu')

# Pulizia di sicurezza nel caso i pesi siano stati salvati con tool esterni
# Rimuove prefissi "module." o "model." se presenti, lasciando le chiavi pulite per timm
state_dict_pulito = {k.replace("module.", "").replace("model.", ""): v for k, v in state_dict.items()}

model.load_state_dict(state_dict_pulito, strict=True)
print("Successo! Nessun mismatch di chiavi o dimensioni.")

# 4. TEST SUL DATASET CIFAR-10
print("\n4. Test su un batch di CIFAR-10...")
# Normalizzazione standard di CIFAR 
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=5, shuffle=False)

model.eval() # Disabilita dropout e batchnorm
immagini, etichette = next(iter(testloader))

with torch.no_grad():
    output = model(immagini)
    _, predizioni = torch.max(output, 1)



1. Download/Verifica del file locale...
File pronto in: modello_edadaltocg/pytorch_model.bin

2. Costruzione della ResNet-50 per CIFAR-10...

3. Iniezione dei pesi...
Successo! Nessun mismatch di chiavi o dimensioni.

4. Test su un batch di CIFAR-10...

--- RISULTATO INFERENZA ---
Immagine 1 | Reale: Gatto   | Predetta: Gatto   | OK
Immagine 2 | Reale: Nave    | Predetta: Nave    | OK
Immagine 3 | Reale: Nave    | Predetta: Nave    | OK
Immagine 4 | Reale: Aereo   | Predetta: Aereo   | OK
Immagine 5 | Reale: Rana    | Predetta: Rana    | OK


In [ ]:
# Set the model to evaluation mode
model.eval()


classi = ['Aereo', 'Auto', 'Uccello', 'Gatto', 'Cervo', 'Cane', 'Rana', 'Cavallo', 'Nave', 'Camion']

print("\n--- RISULTATO INFERENZA ---")
for i in range(5):
    reale = classi[etichette[i]]
    predetta = classi[predizioni[i]]
    print(f"Immagine {i+1} | Reale: {reale:7} | Predetta: {predetta:7} | {'OK' if reale == predetta else 'ERRORE'}")

correct = 0
total = 0

# Disable gradient calculations during inference
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # If you have a GPU, move data to GPU
        # images = images.to('cuda')
        # labels = labels.to('cuda')

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the model on the 10000 test images: {accuracy:.2f}%')

Accuracy of the model on the 10000 test images: 94.65%
